# RSM317 – Group Assignment 1: Cleaning and Parsing SEC Form 10-K Filings

**Team:** Vladislav Glotov (1010946620), Ryosuke Fukui (1013980694), Andrew Ding (1009932218)

We load the latest 10-K (HTML from EDGAR) of ten companies, remove the HTML tags, spell-check the text, and extract the fiscal year end, Item 3, signature dates and signers into `legalproceedings.csv`.

## 1. Loading the filings
We loop over all `*.html` files in the folder. The company name comes from the file name.

In [1]:
import re
from pathlib import Path
from collections import Counter

import pandas as pd

current_directory = Path.cwd()

company_names = []
raw_html = []
for html_file in current_directory.glob("*.html"):
    with html_file.open("r", encoding="utf-8") as file:
        content = file.read()
    company = html_file.name.replace(".html", "").replace("_10-K", "").replace("_", " ")
    company_names.append(company)
    raw_html.append(content)
    print(f"Loaded {html_file.name}: {len(content)} characters")

print(f"\n{len(company_names)} filings loaded")

Loaded Alphabet_10-K.html: 2616614 characters
Loaded Amazon_10-K.html: 1968457 characters
Loaded Apple_10-K.html: 1520323 characters


Loaded Coca-Cola_10-K.html: 3758255 characters
Loaded Johnson_and_Johnson_10-K.html: 3701702 characters
Loaded Meta_10-K.html: 2371865 characters


Loaded Microsoft_10-K.html: 8585616 characters
Loaded Procter_and_Gamble_10-K.html: 2599600 characters
Loaded Tesla_10-K.html: 2391644 characters


Loaded Walmart_10-K.html: 2324096 characters

10 filings loaded


## 2. Removing the HTML tags
Regular expressions and string methods only:
1. Delete the `<script>` block and the hidden `<ix:header>` block with `re.search` and slicing.
2. Replace block-end tags and `<br>` with a newline, and `</td>`/`</th>` with ` | `.
3. Delete all other tags **without inserting a space**, because small capitals are split across tags (Alphabet: `S<span>UNDAR</span>`).
4. Convert HTML entities (`&#8217;`, `&amp;`) back to characters and tidy whitespace.

In [2]:
def remove_block(text, start_tag, end_tag):
    "Remove every block from start_tag up to and including end_tag."
    for _ in range(len(re.findall(start_tag, text, flags=re.I))):
        opening = re.search(start_tag, text, flags=re.I)
        if opening:
            closing = re.search(end_tag, text[opening.end():], flags=re.I)
            if closing:
                end = opening.end() + closing.end()
                text = text[: opening.start()] + " " + text[end:]
    return text


# Numeric HTML entities that appear in these filings, written with the \u escapes from class.
ENTITIES = {
    "&#160;": "\u00a0", "&#165;": "\u00a5", "&#167;": "\u00a7", "&#168;": "\u00a8",
    "&#169;": "\u00a9", "&#174;": "\u00ae", "&#224;": "\u00e0", "&#225;": "\u00e1",
    "&#227;": "\u00e3", "&#231;": "\u00e7", "&#233;": "\u00e9", "&#237;": "\u00ed",
    "&#243;": "\u00f3", "&#253;": "\u00fd", "&#254;": "\u00fe", "&#287;": "\u011f",
    "&#304;": "\u0130", "&#350;": "\u015e", "&#894;": "\u037e", "&#8201;": "\u2009",
    "&#8209;": "\u2011", "&#8210;": "\u2012", "&#8211;": "\u2013", "&#8212;": "\u2014",
    "&#8217;": "\u2019", "&#8220;": "\u201c", "&#8221;": "\u201d", "&#8224;": "\u2020",
    "&#8225;": "\u2021", "&#8226;": "\u2022", "&#8239;": "\u202f", "&#8364;": "\u20ac",
    "&#8482;": "\u2122", "&#9702;": "\u25e6", "&#9744;": "\u2610", "&#9745;": "\u2611",
    "&#9746;": "\u2612",
}


def clean_html(raw):
    "Remove HTML tags with regular expressions and string methods and return readable text."
    text = remove_block(raw, "<script", "</script>")
    text = remove_block(text, "<ix:header", "</ix:header>")          # hidden XBRL data
    text = re.sub(r"<br\s*/?>|</p>|</div>|</tr>|</li>|</h\d>|</table>", "\n", text, flags=re.I)
    text = re.sub(r"</td>|</th>", " | ", text, flags=re.I)
    text = re.sub(r"<[^>]+>", "", text)                               # all other tags, no space
    for entity in ENTITIES:
        text = text.replace(entity, ENTITIES[entity])
    text = text.replace("&amp;", "&")
    text = text.replace("\u00a0", " ").replace("\u200b", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" ?\|[ |]*\| ?", " | ", text)                       # several empty cells -> one separator
    lines = [re.sub(r"^[ |]+|[ |]+$", "", line) for line in text.split("\n")]
    return "\n".join(line for line in lines if line)                  # no empty lines


clean_texts = []
for i in range(len(company_names)):
    company = company_names[i]
    cleaned = clean_html(raw_html[i])
    clean_texts.append(cleaned)
    out_name = company.replace(" ", "_") + "_10-K.txt"
    with open(out_name, "w", encoding="utf-8") as out_file:
        out_file.write(cleaned)
    print(f"{company} {len(raw_html[i])} -> {len(cleaned)} characters saved to {out_name}")

Alphabet 2616614 -> 348278 characters saved to Alphabet_10-K.txt
Amazon 1968457 -> 286487 characters saved to Amazon_10-K.txt
Apple 1520323 -> 209418 characters saved to Apple_10-K.txt


Coca-Cola 3758255 -> 606280 characters saved to Coca-Cola_10-K.txt
Johnson and Johnson 3701702 -> 423480 characters saved to Johnson_and_Johnson_10-K.txt


Meta 2371865 -> 519344 characters saved to Meta_10-K.txt


Microsoft 8585616 -> 322748 characters saved to Microsoft_10-K.txt
Procter and Gamble 2599600 -> 309648 characters saved to Procter_and_Gamble_10-K.txt
Tesla 2391644 -> 399759 characters saved to Tesla_10-K.txt


Walmart 2324096 -> 363456 characters saved to Walmart_10-K.txt


In [3]:
# A quick look at the result: the beginning of Apple's report
apple_text = ""
for i in range(len(company_names)):
    if company_names[i] == "Apple":
        apple_text = clean_texts[i]
print(apple_text[:1500])

aapl-20250927
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27, 2025
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from to .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California | 94-2404110
(State or other jurisdiction
of incorporation or organization)
(I.R.S. Employer Identification No.)
One Apple Park Way
Cupertino, California
95014
(Address of principal executive offices) | (Zip Code)
(408) 996-1010
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
Title of each class | Trading symbol(s) | Name of each exchange on which registered
Common Stock, $0.00001 par value per share
AAPL | The Nasdaq Stock Market LLC
0.000% Notes due

## 3. Spell checking with Norvig's spell checker
Corpus: Webster's Unabridged Dictionary (Gutenberg header and licence removed).

In [4]:
webster_file = list(current_directory.glob("Webster*.txt"))[0]
with webster_file.open("r", encoding="utf-8") as file:
    webster = file.read()
webster = webster.split("*** START OF")[-1]
lines = webster.split("\n")
webster = "\n".join(lines[1:])
webster = webster.split("*** END OF")[0]


# ----- Norvig's spell checker (unchanged, only the corpus is different) -----
def words(text):
    return re.findall(r"\w+", text.lower())

WORDS = Counter(words(webster))

def P(word, N=sum(WORDS.values())):
    "Probability of `word`."
    return WORDS[word] / N

def correction(word):
    "Most probable spelling correction for word."
    return max(candidates(word), key=P)

def candidates(word):
    "Generate possible spelling corrections for word."
    return known([word]) or known(edits1(word)) or known(edits2(word)) or {word}

def known(words):
    "The subset of `words` that appear in the dictionary of WORDS."
    return set(w for w in words if w in WORDS)

def edits1(word):
    "All edits that are one edit away from `word`."
    letters    = "abcdefghijklmnopqrstuvwxyz"
    splits     = [(word[:i], word[i:])    for i in range(len(word) + 1)]
    deletes    = [L + R[1:]               for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces   = [L + c + R[1:]           for L, R in splits if R for c in letters]
    inserts    = [L + c + R               for L, R in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)

def edits2(word):
    "All edits that are two edits away from `word`."
    return (e2 for e1 in edits1(word) for e2 in edits1(e1))


print(f"Webster's corpus: {len(WORDS)} different words, {sum(WORDS.values())} words in total")

Webster's corpus: 211279 different words, 4734121 words in total


### 3.1 A naive first run
For each filing we count the distinct words that are not in the dictionary.

In [5]:
naive_rows = []
for i in range(len(company_names)):
    unique_tokens = set(words(clean_texts[i]))
    unknown = [w for w in unique_tokens if w not in WORDS]
    naive_rows.append({"company": company_names[i], "unique_tokens": len(unique_tokens), "unknown_naive": len(unknown)})

naive_df = pd.DataFrame(naive_rows)
naive_df

,company,unique_tokens,unknown_naive
0,Alphabet,4317,873
1,Amazon,3830,821
2,Apple,3346,687
3,Coca-Cola,5348,1299
4,Johnson and Johnson,5104,1296
5,Meta,4580,803
6,Microsoft,4244,907
7,Procter and Gamble,4053,876
8,Tesla,4602,892
9,Walmart,4753,982


In [6]:
# What does the checker suggest for some typical 10-K words that are spelled correctly?
test_words = ["cybersecurity", "smartphone", "iphone", "online", "website", "ebitda",
              "blockchain", "nasdaq", "subsidiaries", "fiscal", "litigation", "covid"]

for w in test_words:
    print(f"{w} in Webster? {str(w in WORDS)} -> {correction(w)}")

cybersecurity in Webster? False -> cybersecurity
smartphone in Webster? False -> smartphone
iphone in Webster? False -> phone
online in Webster? False -> unline
website in Webster? False -> webster
ebitda in Webster? False -> bita


blockchain in Webster? False -> blockchain
nasdaq in Webster? False -> nasal
subsidiaries in Webster? True -> subsidiaries
fiscal in Webster? True -> fiscal
litigation in Webster? True -> litigation
covid in Webster? False -> coid


In [7]:
# What does the original checker do to ALL unknown words of one report (Apple)?
apple_text = ""
for i in range(len(company_names)):
    if company_names[i] == "Apple":
        apple_text = clean_texts[i]
apple_unknown = [w for w in set(words(apple_text)) if w not in WORDS]
apple_corrections = [(w, correction(w)) for w in apple_unknown]
changed = sum(1 for w, c in apple_corrections if w != c)
print(f"{len(apple_unknown)} unknown words in Apple's 10-K; {changed} would be changed")
print("Some of them:", apple_corrections[:15])

687 unknown words in Apple's 10-K; 641 would be changed
Some of them: [('369', '36'), ('357', '37'), ('977', '97'), ('aspx', 'asp'), ('electronically', 'electrically'), ('2051', '251'), ('behaviors', 'behavior'), ('909', '90'), ('smartwatches', 'smartwatches'), ('848', '48'), ('ipad', 'pad'), ('apps', 'alps'), ('2053', '253'), ('309', '30'), ('ipados', 'spades')]


### 3.2 Challenges
1. **A dictionary is not normal text:** word frequencies come from the definitions, so the candidates are ranked poorly.
2. **Old vocabulary:** the 1913 edition has no modern words (*online*, *cybersecurity*).
3. **Names, brands and acronyms** (*iPhone*, *EBITDA*) are not in any dictionary.
4. **Numbers** count as words and get "corrected" (`010` → `10`).
5. **Inflected forms** are missing (*amortizing*).
6. **Speed:** `edits2` is too slow for whole reports, let alone the 2 TB archive.
7. **Wrong corrections** damage correct finance terms.

### 3.3 How we address them
- Keep Norvig's `correction` exactly as in class, including `edits2`. The sample words and Apple's report show what it does.
- Before counting the remaining unknown words, apply the cleaning steps from class: keep alphabetic tokens longer than one letter (`isalpha`, `len`). That drops numbers such as `010`.
- Accept a word when it is in Webster or when its Porter stem is in Webster.
- Lancaster is not used for that test. In class it stems *better* to *bet*, so it would treat real errors as known words.
- Only **flag** possible errors. The clean text is never overwritten.

In [8]:
from nltk.stem import PorterStemmer, LancasterStemmer

porter = PorterStemmer()
lancaster = LancasterStemmer()

print("Lancaster stems 'better' to", lancaster.stem('better'))
print("Porter stems 'better' to", porter.stem('better'))


def lecture_known(word):
    "Known if the word or its Porter stem is in Webster."
    return word in WORDS or porter.stem(word) in WORDS


def tokens_to_check(text):
    "Alphabetic words longer than one letter. words() already lower-cases them."
    return set(word for word in words(text) if len(word) > 1 and word.isalpha())


improved_counts, examples = [], []
for i in range(len(company_names)):
    flagged = [word for word in tokens_to_check(clean_texts[i]) if not lecture_known(word)]
    improved_counts.append(len(flagged))
    short = [word for word in flagged if len(word) <= 8]
    for word in short[:3]:
        examples.append({"company": company_names[i], "flagged word": word, "suggestion": correction(word)})

spell_rows = []
for i in range(len(naive_rows)):
    row = naive_rows[i]
    spell_rows.append([row["company"], row["unique_tokens"], row["unknown_naive"], improved_counts[i]])
spell_df = pd.DataFrame(
    spell_rows,
    columns=["company", "unique_tokens", "unknown_naive", "unknown_improved"],
)
spell_df


Lancaster stems 'better' to bet
Porter stems 'better' to better


,company,unique_tokens,unknown_naive,unknown_improved
0,Alphabet,4317,873,274
1,Amazon,3830,821,233
2,Apple,3346,687,202
3,Coca-Cola,5348,1299,469
4,Johnson and Johnson,5104,1296,533
5,Meta,4580,803,322
6,Microsoft,4244,907,287
7,Procter and Gamble,4053,876,277
8,Tesla,4602,892,340
9,Walmart,4753,982,383


In [9]:
# Examples of words that are still flagged, with the checker's suggestion
pd.DataFrame(examples)

,company,flagged word,suggestion
0,Alphabet,redeploy,deploy
1,Alphabet,blogs,blows
2,Alphabet,webcast,recast
3,Amazon,hogan,cogan
4,Amazon,prc,pr
5,Amazon,pcaob,prob
6,Apple,aspx,asp
7,Apple,ipad,pad
8,Apple,ipados,spades
9,Coca-Cola,cchbc,chic


### 3.4 Result
The naive checker flags 687–1,299 distinct tokens per report. After the alphabetic filter and Porter stemming, 202–533 words per report are still unknown. Almost all of them are names, brands or modern technical terms. Norvig's suggestions for those words, such as *admob* → *armor*, are not usable, which is why the clean text is never overwritten.

## 4. Extracting the information
Each function runs regexes on the clean text and returns `""` if it finds nothing. Dates are converted to ISO format with string methods (`split`, `replace`, and a month list).

In [10]:
MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]


def to_iso(date_str):
    "'September 27 , 2025' -> '2025-09-27'"
    date_str = date_str.replace(",", " ")
    date_str = re.sub(r"\s+", " ", date_str.strip())
    parts = date_str.split(" ")
    month_name = parts[0]
    day = parts[1]
    year = parts[2]
    month_number = ""
    for i, month in enumerate(MONTHS):
        if month.lower() == month_name.lower():
            month_number = str(i + 1)
    if len(month_number) == 1:
        month_number = "0" + month_number
    if len(day) == 1:
        day = "0" + day
    return year + "-" + month_number + "-" + day


print(to_iso("September 27 , 2025"), to_iso("JUNE 30, 2026"))

2025-09-27 2026-06-30


**1. Fiscal year end:** the first "fiscal year ended *date*", which is on the cover page.

In [11]:
def get_fiscal_year_end(text):
    best = None
    for month in MONTHS:
        pattern = r"fiscal\s+year\s+ended\s*:?\s*" + month + r"\s+\d\d?\s*,\s*\d\d\d\d"
        match = re.search(pattern, text, flags=re.I)
        if match and (best is None or match.start() < best.start()):
            best = match
    if not best:
        return ""
    date_str = re.sub(r"fiscal\s+year\s+ended\s*:?\s*", "", best.group(0), flags=re.I)
    return to_iso(date_str)

**2. Item 3:** from the last "Item 3 … Legal Proceedings" heading (the first one is in the table of contents) to the next line starting with "Item 4".

In [12]:
def get_item3(text):
    patterns = [
        r"\n\s*item\s*3\s*[.:]?\s*\|\s*legal\s+proceedings\.?",
        r"\n\s*item\s*3\s*[.:]?\s*legal\s+proceedings\.?",
    ]
    starts = [match for match in re.finditer(patterns[0], text, flags=re.I)]
    starts = starts + [match for match in re.finditer(patterns[1], text, flags=re.I)]
    if not starts:
        return ""
    start_match = starts[0]
    for match in starts:
        if match.start() > start_match.start():
            start_match = match
    start = start_match.end()   # later heading, not the table of contents
    end = re.search(r"\n\s*item\s*4\D", text[start:], flags=re.I)
    if end:
        section = text[start : start + end.start()]
    else:
        section = text[start:]
    section = re.sub(r"^[ .|\n]+", "", section)
    return re.sub(r"[ |\n]+$", "", section)

**3–4. Signatures:** the section from the last "SIGNATURES" line to the exhibit index.
- **Dates:** every date in the section, including the form "28th day of January 2026" (Meta). Walmart's "Fiscal Year Ended" footer is removed first.
- **Signers:** the names after `/s/`. Upper-case names are replaced by the normally printed version in the filing (*McCarthy*, not *Mccarthy*). A name directly followed by a "Director" line also counts, which catches Coca-Cola's attorney-in-fact signatures. We remove the comma before "Jr.", drop "LLP", and remove duplicates.

In [13]:
def get_signature_section(text):
    heads = list(re.finditer(r"\nsignatures\n", text, flags=re.I))
    if not heads:
        return ""
    section = text[heads[-1].start() + 1:]
    exhibit = re.search(r"\n\s*exhibit\s+index", section, flags=re.I)
    if exhibit:
        return section[: exhibit.start()]
    return section


def ordinal_to_iso(phrase):
    "'28th day of January, 2026' -> '2026-01-28'"
    phrase = phrase.replace(",", " ")
    phrase = re.sub(r"\s+", " ", phrase.strip())
    parts = phrase.split(" ")
    day = parts[0]
    if day.lower()[-2:] == "st" or day.lower()[-2:] == "nd" or day.lower()[-2:] == "rd" or day.lower()[-2:] == "th":
        day = day[:-2]
    return to_iso(parts[3] + " " + day + ", " + parts[4])


def get_signature_dates(section):
    for month in MONTHS:
        section = re.sub(
            r"fiscal\s+year\s+ended\s*" + month + r"\s+\d\d?\s*,\s*\d\d\d\d",
            " ",
            section,
            flags=re.I,
        )
    found = []
    for month in MONTHS:
        for match in re.finditer(month + r"\s+\d\d?\s*,\s*\d\d\d\d", section, flags=re.I):
            date = to_iso(match.group(0))
            if not date in found:
                found.append(date)
        for suffix in ["st", "nd", "rd", "th"]:
            pattern = r"\d\d?" + suffix + r"\s+day\s+of\s+" + month + r"\s*,?\s*\d\d\d\d"
            for match in re.finditer(pattern, section, flags=re.I):
                date = ordinal_to_iso(match.group(0))
                if not date in found:
                    found.append(date)
    return ", ".join(found)


def plain_letters(name):
    "Lower case without accents and brackets: 'Ch\u00e1vez' -> 'chavez', so names can be compared."
    name = re.sub(r"^[ ()]+|[ ()]+$", "", name).lower()
    for accented, plain in [
        ("\u00e1", "a"), ("\u00e0", "a"), ("\u00e2", "a"), ("\u00e4", "a"), ("\u00e3", "a"),
        ("\u00e9", "e"), ("\u00e8", "e"), ("\u00ea", "e"), ("\u00eb", "e"),
        ("\u00ed", "i"), ("\u00ec", "i"), ("\u00ee", "i"), ("\u00ef", "i"),
        ("\u00f3", "o"), ("\u00f2", "o"), ("\u00f4", "o"), ("\u00f6", "o"), ("\u00f5", "o"),
        ("\u00fa", "u"), ("\u00f9", "u"), ("\u00fb", "u"), ("\u00fc", "u"),
        ("\u00f1", "n"), ("\u00e7", "c"),
    ]:
        name = name.replace(accented, plain)
    return name


def drop_comma_before_suffix(name):
    "'Moritz, Jr.' -> 'Moritz Jr.' without using a backreference."
    for suffix in ["Jr.", "Sr.", "II", "III"]:
        tail = ", " + suffix
        if len(name) >= len(tail) and name.lower()[-len(tail):] == tail.lower():
            original_suffix = name[-len(suffix):]
            return name[:-len(tail)] + " " + original_suffix
    return name


def get_signers(section):
    names = []
    for match in re.finditer(r"/\s*s\s*/\s*[^|\n/]+", section, flags=re.I):
        names.append(re.sub(r"^/\s*s\s*/\s*", "", match.group(0), flags=re.I))

    # directors who signed through an attorney-in-fact: a name followed by a line "Director"
    lines = section.split("\n")
    for i in range(len(lines) - 1):
        next_line = lines[i + 1]
        if next_line == "Director" or next_line[:9] == "Director " or next_line[:9] == "Director,":
            candidate = lines[i].split("|")[-1].strip()
            parts = candidate.split()
            starts_with_capital = True
            for part in parts:
                if part[0] != part[0].upper() or not part[0].isalpha():
                    starts_with_capital = False
            if len(parts) > 1:
                if not len(parts) > 4:
                    if starts_with_capital:
                        names.append(candidate)

    # all lines / table cells of the section, used to find the normally printed version of a name
    cells = [re.sub(r"^[ ()]+|[ ()]+$", "", c) for c in section.replace("|", "\n").split("\n")]

    signers = []
    for name in names:
        name = re.sub(r"^[ ,*]+|[ ,*]+$", "", name)
        name = drop_comma_before_suffix(name)
        if name and "LLP" not in name.upper():
            if name == name.upper():
                printed = [c for c in cells if c != c.upper() and plain_letters(c) == plain_letters(name)]
                if printed:
                    name = printed[0]
                else:
                    name = name.title()
            if plain_letters(name) not in [plain_letters(s) for s in signers]:
                signers.append(name)
    return ", ".join(signers)

We run all functions for every company, collect the results in a dictionary, and build the DataFrame from it.

In [14]:
results = {"company": [], "fiscal_year_end": [], "legal_proceedings": [], "signature_dates": [], "signers": []}

for i in range(len(company_names)):
    text = clean_texts[i]
    signature_section = get_signature_section(text)
    results["company"].append(company_names[i])
    results["fiscal_year_end"].append(get_fiscal_year_end(text))
    results["legal_proceedings"].append(get_item3(text))
    results["signature_dates"].append(get_signature_dates(signature_section))
    results["signers"].append(get_signers(signature_section))

df = pd.DataFrame(results)
df.to_csv("legalproceedings.csv", index=False)

# show the table with the length of Item 3 instead of the full text
pd.set_option("max_colwidth", 400)
overview = pd.DataFrame(
    [[df["company"][i], df["fiscal_year_end"][i], len(df["legal_proceedings"][i]),
      df["signature_dates"][i], df["signers"][i]] for i in range(len(df))],
    columns=["company", "fiscal_year_end", "item3_length", "signature_dates", "signers"],
)
overview

,company,fiscal_year_end,item3_length,signature_dates,signers
0,Alphabet,2025-12-31,246,2026-02-04,"Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, K. Ram Shriram, Robin L. Washington"
1,Amazon,2025-12-31,130,2026-02-05,"Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks"
2,Apple,2025-09-27,5372,2025-10-31,"Kevan Parekh, Timothy D. Cook, Chris Kondo, Wanda Austin, Alex Gorsky, Andrea Jung, Arthur D. Levinson, Monica Lozano, Ronald D. Sugar, Susan L. Wagner"
3,Coca-Cola,2025-12-31,22067,2026-02-20,"James Quincey, John Murphy, Erin L. May, Jennifer D. Manning, Herb Allen, Christopher C. Davis, Bela Bajaria, Carolyn Everson, Ana Botín, Thomas S. Gayner, Maria Elena Lagomasino, Caroline J. Tsay, Max Levchin, David B. Weinberg, Amity Millhiser"
4,Johnson and Johnson,2025-12-28,218,2026-02-11,"J. Duato, J. J. Wolk, R. J. Decker Jr., M. C. Beckerle, J. A. Doudna, M. A. Hewson, P. A. Johnson, H. Joly, M. B. McClellan, J. G. Morikis, D. E. Pinto, M. A. Weinberger, N. Y. West, E. A. Woods"
5,Meta,2025-12-31,38480,2026-01-28,"Susan Li, Mark Zuckerberg, Aaron Anderson, Peggy Alford, Marc L. Andreessen, John Arnold, Patrick Collison, John Elkann, Andrew W. Houston, Nancy Killefer, Robert M. Kimmitt, Charles Songhurst, Hock E. Tan, Tracey T. Travis, Dana White, Tony Xu"
6,Microsoft,2026-06-30,175,2026-07-29,"Alice L. Jolla, Satya Nadella, Carmine Di Sibio, Reid G. Hoffman, Hugh F. Johnston, Teri L. List, Catherine MacGregor, Mark A. L. Mason, Sandra E. Peterson, Penny S. Pritzker, John David Rainey, Charles W. Scharf, John W. Stanton, Emma N. Walmsley, Amy E. Hood"
7,Procter and Gamble,2026-06-30,1852,2026-08-04,"Shailesh Jejurikar, Andre Schulten, Matthew W. Janzaruk, B. Marc Allen, Craig Arnold, Brett Biggs, Sheila Bonini, Amy L. Chang, Joseph Jimenez, Christopher J. Kempczinski, Debra L. Lee, Christine M. McCarthy, Ashley McEvoy, Robert J. Portman, Rajesh Subramaniam"
8,Tesla,2025-12-31,200,2026-01-28,"Elon Musk, Vaibhav Taneja, Robyn Denholm, Ira Ehrenpreis, Joseph Gebbia, Jack Hartung, James Murdoch, Kimbal Musk, JB Straubel, Kathleen Wilson-Thompson"
9,Walmart,2026-01-31,5390,2026-03-13,"John R. Furner, Gregory B. Penner, John David Rainey, Dwayne M. Milum, Cesar Conde, Timothy P. Flynn, Sarah Friar, Carla A. Harris, Thomas W. Horton, Marissa A. Mayer, C. Douglas McMillon, Shishir Mehrotra, Robert E. Moritz Jr., Brian Niccol, Randall L. Stephenson, Steuart L. Walton"


## 5. Verification
### (a) Start and end of Item 3
We print the first and last 200 characters of each section and check that no "Item 4" heading is included.

In [15]:
for i in range(len(results["company"])):
    company = results["company"][i]
    text = results["legal_proceedings"][i]
    has_item4 = re.search(r"\n\s*item\s*4\D", "\n" + text + " ", flags=re.I) is not None
    print("=" * 100)
    print(f"{company} ({len(text)} characters) contains an 'Item 4' heading: {has_item4}")
    print("FIRST 200:", text[:200])
    print("LAST 200:", text[-200:])

Alphabet (246 characters) contains an 'Item 4' heading: False
FIRST 200: For a description of our material pending legal proceedings, see Legal Matters in Note 10 of the Notes to Consolidated Financial Statements included in Part II, Item 8 of this Annual Report on Form 10
LAST 200: l proceedings, see Legal Matters in Note 10 of the Notes to Consolidated Financial Statements included in Part II, Item 8 of this Annual Report on Form 10-K, which is incorporated herein by reference.
Amazon (130 characters) contains an 'Item 4' heading: False
FIRST 200: See Item 8 of Part II, “Financial Statements and Supplementary Data — Note 7 — Commitments and Contingencies — Legal Proceedings.”
LAST 200: See Item 8 of Part II, “Financial Statements and Supplementary Data — Note 7 — Commitments and Contingencies — Legal Proceedings.”
Apple (5372 characters) contains an 'Item 4' heading: False
FIRST 200: Digital Markets Act Investigations
On March 25, 2024, the Commission announced that it had opened a f

### (b) Check table
`manual` holds the values we read in each filing. Signers are compared as sets: order is ignored, but spelling, accents and capitalisation must match.

In [16]:
manual = {
    "Apple": {
        "fiscal_year_end": "2025-09-27", "signature_dates": "2025-10-31",
        "signers": "Kevan Parekh, Timothy D. Cook, Chris Kondo, Wanda Austin, Alex Gorsky, Andrea Jung, "
                   "Arthur D. Levinson, Monica Lozano, Ronald D. Sugar, Susan L. Wagner"},
    "Microsoft": {
        "fiscal_year_end": "2026-06-30", "signature_dates": "2026-07-29",
        "signers": "Alice L. Jolla, Satya Nadella, Carmine Di Sibio, Reid G. Hoffman, Hugh F. Johnston, "
                   "Teri L. List, Catherine MacGregor, Mark A. L. Mason, Sandra E. Peterson, Penny S. Pritzker, "
                   "John David Rainey, Charles W. Scharf, John W. Stanton, Emma N. Walmsley, Amy E. Hood"},
    "Amazon": {
        "fiscal_year_end": "2025-12-31", "signature_dates": "2026-02-05",
        "signers": "Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, "
                   "Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, "
                   "Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks"},
    "Tesla": {
        "fiscal_year_end": "2025-12-31", "signature_dates": "2026-01-28",
        "signers": "Elon Musk, Vaibhav Taneja, Robyn Denholm, Ira Ehrenpreis, Joseph Gebbia, Jack Hartung, "
                   "James Murdoch, Kimbal Musk, JB Straubel, Kathleen Wilson-Thompson"},
    "Meta": {
        "fiscal_year_end": "2025-12-31", "signature_dates": "2026-01-28",
        "signers": "Susan Li, Mark Zuckerberg, Aaron Anderson, Peggy Alford, Marc L. Andreessen, John Arnold, "
                   "Patrick Collison, John Elkann, Andrew W. Houston, Nancy Killefer, Robert M. Kimmitt, "
                   "Charles Songhurst, Hock E. Tan, Tracey T. Travis, Dana White, Tony Xu"},
    "Alphabet": {
        "fiscal_year_end": "2025-12-31", "signature_dates": "2026-02-04",
        "signers": "Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, "
                   "R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, "
                   "K. Ram Shriram, Robin L. Washington"},
    "Johnson and Johnson": {
        "fiscal_year_end": "2025-12-28", "signature_dates": "2026-02-11",
        "signers": "J. Duato, J. J. Wolk, R. J. Decker Jr., M. C. Beckerle, J. A. Doudna, M. A. Hewson, "
                   "P. A. Johnson, H. Joly, M. B. McClellan, J. G. Morikis, D. E. Pinto, M. A. Weinberger, "
                   "N. Y. West, E. A. Woods"},
    "Coca-Cola": {
        "fiscal_year_end": "2025-12-31", "signature_dates": "2026-02-20",
        "signers": "James Quincey, John Murphy, Erin L. May, Herb Allen, Christopher C. Davis, Bela Bajaria, "
                   "Carolyn Everson, Ana Botín, Thomas S. Gayner, Maria Elena Lagomasino, Caroline J. Tsay, "
                   "Max Levchin, David B. Weinberg, Amity Millhiser, Jennifer D. Manning"},
    "Procter and Gamble": {
        "fiscal_year_end": "2026-06-30", "signature_dates": "2026-08-04",
        "signers": "Shailesh Jejurikar, Andre Schulten, Matthew W. Janzaruk, B. Marc Allen, Craig Arnold, "
                   "Brett Biggs, Sheila Bonini, Amy L. Chang, Joseph Jimenez, Christopher J. Kempczinski, "
                   "Debra L. Lee, Christine M. McCarthy, Ashley McEvoy, Robert J. Portman, Rajesh Subramaniam"},
    "Walmart": {
        "fiscal_year_end": "2026-01-31", "signature_dates": "2026-03-13",
        "signers": "John R. Furner, Gregory B. Penner, John David Rainey, Dwayne M. Milum, Cesar Conde, "
                   "Timothy P. Flynn, Sarah Friar, Carla A. Harris, Thomas W. Horton, Marissa A. Mayer, "
                   "C. Douglas McMillon, Shishir Mehrotra, Robert E. Moritz Jr., Brian Niccol, "
                   "Randall L. Stephenson, Steuart L. Walton"},
}


def name_set(names):
    return {" ".join(n.split()) for n in names.split(",") if n.strip()}


def same_names(a, b):
    return name_set(a) == name_set(b)


check_rows = []
for i, company in enumerate(results["company"]):
    truth = manual[company]
    for field in ["fiscal_year_end", "signature_dates", "signers"]:
        extracted = results[field][i]
        if field == "signers":
            ok = same_names(extracted, truth[field])
        else:
            ok = extracted == truth[field]
        match = "✘ MISMATCH"
        if ok:
            match = "✔"
        check_rows.append({"company": company, "field": field, "extracted": extracted,
                           "manual (read by us)": truth[field], "match": match})

check_df = pd.DataFrame(check_rows)
pd.set_option("max_colwidth", 400)   # show the signer lists
print(Counter(check_df["match"]))
check_df

Counter({'✔': 30})


,company,field,extracted,manual (read by us),match
0,Alphabet,fiscal_year_end,2025-12-31,2025-12-31,✔
1,Alphabet,signature_dates,2026-02-04,2026-02-04,✔
2,Alphabet,signers,"Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, K. Ram Shriram, Robin L. Washington","Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, K. Ram Shriram, Robin L. Washington",✔
3,Amazon,fiscal_year_end,2025-12-31,2025-12-31,✔
4,Amazon,signature_dates,2026-02-05,2026-02-05,✔
5,Amazon,signers,"Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks","Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks",✔
6,Apple,fiscal_year_end,2025-09-27,2025-09-27,✔
7,Apple,signature_dates,2025-10-31,2025-10-31,✔
8,Apple,signers,"Kevan Parekh, Timothy D. Cook, Chris Kondo, Wanda Austin, Alex Gorsky, Andrea Jung, Arthur D. Levinson, Monica Lozano, Ronald D. Sugar, Susan L. Wagner","Kevan Parekh, Timothy D. Cook, Chris Kondo, Wanda Austin, Alex Gorsky, Andrea Jung, Arthur D. Levinson, Monica Lozano, Ronald D. Sugar, Susan L. Wagner",✔
9,Coca-Cola,fiscal_year_end,2025-12-31,2025-12-31,✔


In [17]:
# Details of every mismatch
for r in check_rows:
    if r["match"] != "✔":
        if r["field"] == "signers":
            manual_names = name_set(r["manual (read by us)"])
            extracted_names = name_set(r["extracted"])
            missing = [name for name in manual_names if not name in extracted_names]
            extra = [name for name in extracted_names if not name in manual_names]
            print(f"{r['company']} signers  missing: {missing}  extra: {extra}")
        else:
            print(f"{r['company']} {r['field']}: extracted '{r['extracted']}', filing says '{r['manual (read by us)']}'")

**Result of (a).** No section contains an "Item 4" heading, and in every case the next line in the clean text is the Item 4 heading.

- **Alphabet, Amazon, J&J, Microsoft, Tesla:** one sentence referring to a note in Item 8 ✔
- **Apple:** "Digital Markets Act Investigations" → last paragraph on possible losses ✔
- **Coca-Cola:** "The Company is involved in various legal proceedings…" → "…strong defenses to the claims." ✔
- **Meta:** "As a multinational company…" → "…legal proceedings and disputes in the future." ✔
- **P&G:** "The Company is subject, from time to time…" → reference to Item 1A ✔
- **Walmart:** "I. SUPPLEMENTAL INFORMATION" → last paragraph on possible adverse effects ✔

### (c) Mismatches
All 30 values match after these fixes:
- **Coca-Cola signers:** our first version missed the 11 directors, who are marked `*` and signed through an attorney-in-fact. We fixed this with the general "name followed by Director" rule, which changes nothing for the other companies. Limitation: an attorney-in-fact signer with a title other than "Director" would still be missed.
- **Capitalisation:** `title()` produced "Mccarthy" and "Chavez", and our lower-case comparison hid this. We fixed it by using the printed name and comparing case-sensitively.

Other bugs fixed during development:
- Small capitals split by tags (Alphabet, Microsoft): tags are now removed without a space.
- The table of contents matched first: we now take the last heading.
- Walmart's footer date counted as a signature date: "fiscal year ended + date" is now removed.
- J&J's exhibit index dates: the section is now cut at "Exhibit index".
- The comma in "Robert E. Moritz, Jr." split one name into two: the comma before "Jr." is now removed.
- The auditor's signature is never in the Signatures section. The "LLP" filter is an extra safety check.

## Bonus: Beautiful Soup
We remove the same hidden elements, use `get_text("\n")`, and run the same extraction functions.


In [18]:
from bs4 import BeautifulSoup


def clean_with_bs4(raw):
    soup = BeautifulSoup(raw, "html.parser")
    for tag in soup(["script", "style", "ix:header"]):
        tag.decompose()
    text = soup.get_text("\n").replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()


def plain(text):
    "Item 3 text without table separators and extra spaces, so both versions can be compared."
    return " ".join(text.replace("|", " ").split())


bonus_rows = []
for i in range(len(company_names)):
    company = company_names[i]
    text_bs = clean_with_bs4(raw_html[i])
    section_bs = get_signature_section(text_bs)
    bs4_values = {
        "fiscal_year_end": get_fiscal_year_end(text_bs),
        "legal_proceedings": get_item3(text_bs),
        "signature_dates": get_signature_dates(section_bs),
        "signers": get_signers(section_bs),
    }
    for field in bs4_values:
        bs_value = bs4_values[field]
        rx_value = results[field][i]
        if field == "legal_proceedings":
            agree = plain(rx_value) == plain(bs_value)
            rx_ok = bs_ok = "see (a)"
            rx_show, bs_show = f"{len(plain(rx_value))} chars", f"{len(plain(bs_value))} chars"
        else:
            truth = manual[company][field]
            if field == "signers":
                agree = same_names(rx_value, bs_value)
                rx_ok, bs_ok = same_names(rx_value, truth), same_names(bs_value, truth)
            else:
                agree = rx_value == bs_value
                rx_ok, bs_ok = rx_value == truth, bs_value == truth
            rx_show, bs_show = rx_value, bs_value
        bonus_rows.append({"company": company, "field": field, "regex": rx_show, "bs4": bs_show,
                           "agree": "✔" if agree else "✘", "regex correct": rx_ok, "bs4 correct": bs_ok})

bonus_df = pd.DataFrame(bonus_rows)
print(Counter(bonus_df["agree"]))
pd.DataFrame([r for r in bonus_rows if r["agree"] == "✘"])

Counter({'✔': 33, '✘': 7})


,company,field,regex,bs4,agree,regex correct,bs4 correct
0,Alphabet,signers,"Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, K. Ram Shriram, Robin L. Washington","S, A, F, R. M, L. J, R, J, L, K. R",✘,True,False
1,Coca-Cola,legal_proceedings,22067 chars,22087 chars,✘,see (a),see (a)
2,Meta,legal_proceedings,38480 chars,38526 chars,✘,see (a),see (a)
3,Microsoft,signature_dates,2026-07-29,"2024-01-19, 2026-02-19, 2025-02-20, 2027-02-28, 2011-02-08, 2015-02-12, 2017-02-06, 2014-02-03, 2024-03-08, 2027-03-31, 2026-03-12, 2026-03-10, 2025-03-13, 2025-03-11, 2027-03-05, 2021-03-17, 2026-04-01, 2026-04-30, 2026-05-01, 2026-05-31, 2025-05-01, 2024-05-01, 2026-05-21, 2025-05-15, 2009-05-18, 2013-05-02, 2017-05-26, 2026-06-30, 2026-06-01, 2026-06-10, 2025-06-30, 2024-06-30, 2027-06-30, ...",✘,True,False
4,Microsoft,signers,"Alice L. Jolla, Satya Nadella, Carmine Di Sibio, Reid G. Hoffman, Hugh F. Johnston, Teri L. List, Catherine MacGregor, Mark A. L. Mason, Sandra E. Peterson, Penny S. Pritzker, John David Rainey, Charles W. Scharf, John W. Stanton, Emma N. Walmsley, Amy E. Hood","d, a, S, c, R, H, T, M, P, J, E",✘,True,False
5,Tesla,legal_proceedings,200 chars,201 chars,✘,see (a),see (a)
6,Walmart,legal_proceedings,5390 chars,139663 chars,✘,see (a),see (a)


In [19]:
bonus_df

,company,field,regex,bs4,agree,regex correct,bs4 correct
0,Alphabet,fiscal_year_end,2025-12-31,2025-12-31,✔,True,True
1,Alphabet,legal_proceedings,246 chars,246 chars,✔,see (a),see (a)
2,Alphabet,signature_dates,2026-02-04,2026-02-04,✔,True,True
3,Alphabet,signers,"Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, K. Ram Shriram, Robin L. Washington","S, A, F, R. M, L. J, R, J, L, K. R",✘,True,False
4,Amazon,fiscal_year_end,2025-12-31,2025-12-31,✔,True,True
5,Amazon,legal_proceedings,130 chars,130 chars,✔,see (a),see (a)
6,Amazon,signature_dates,2026-02-05,2026-02-05,✔,True,True
7,Amazon,signers,"Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks","Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks",✔,True,True
8,Apple,fiscal_year_end,2025-09-27,2025-09-27,✔,True,True
9,Apple,legal_proceedings,5372 chars,5372 chars,✔,see (a),see (a)


### Where do the two versions disagree, and which one is right?
All fiscal-year-end dates agree. There are 7 disagreements:

| Company / field | Beautiful Soup problem | Right |
|---|---|---|
| Alphabet – signers | Small-capital names are split across lines, so only "S" is found | regex |
| Microsoft – signers, dates | "SIGNAT\nURES" is not matched, so the section is almost the whole report; names are split as for Alphabet | regex |
| Walmart – Item 3 | Linked cross-references look like headings, so the section runs to the end of the report | regex |
| Coca-Cola, Meta, Tesla – Item 3 | Same content with extra spaces ("$ 3.3 billion") | regex |

**Conclusion:** Beautiful Soup is the more robust HTML parser, but any `get_text` separator either splits words across tags or glues neighbouring cells together. Our regex version treats block, cell and inline tags differently, so it is right in every disagreement.

## AI Appendix

**Tool used:** Claude (Anthropic), through Claude Code.

**What it was used for:** We wrote the notebook first, including the places we had only sketched in comments. Claude Code then turned those comments into working Python and cleaned code that already ran, so the notebook would run from top to bottom. We used the same four steps on every section. This appendix follows one excerpt through all four: the signer names in `get_signers`, and the two helpers that excerpt uses, `drop_comma_before_suffix` and `plain_letters`.

**What we did ourselves:**

1. We wrote the loading loop, the HTML cleaning, the call to Norvig's checker, the extraction functions, the check table, and the explanations. Where a rule was settled but the code was not, we left a comment that stated the rule.
2. We asked Claude Code to replace those comments with working Python and to clean the surrounding code.
3. We read every function it returned. Where we could not explain a line with the string methods, regular expressions, and stemming from class, we rewrote it. Beautiful Soup is only in the bonus. Norvig's functions are the class version.
4. We typed the fiscal year end, the signature dates, and the signers off each filing ourselves, ran the check table, and fixed the rows that failed.

### Worked example: the signer names

Most `/s/` lines are an ordinary name, and those we had working on our own. Four filings are not that simple.

- Procter & Gamble prints `/s/ CHRISTINE M. MCCARTHY` and, on the next line, `(Christine M. McCarthy)`.
- Alphabet prints `/S/ R. MARTIN CHAVEZ` and, under it, `R. Martin Chávez`. The same page has `/S/ ROGER W. FERGUSON JR.`, with no comma.
- Walmart prints `/s/ Robert E. Moritz, Jr.` The comma is part of one name.
- Coca-Cola puts only four names on an `/s/` line: James Quincey, John Murphy, Erin L. May, and Jennifer D. Manning, the attorney-in-fact. The other eleven directors are a name followed by a line that says `Director`, with a `*` where the signature would be.

#### 1. Our draft

The `/s/` loop and the LLP filter were finished code. The three cases above were comments.

```python
def get_signers(section):
    names = []
    for match in re.finditer(r"/s/\s*([^|\n]+)", section, flags=re.I):
        name = match.group(1).strip(" ,*")
        if "LLP" not in name.upper():
            names.append(name)

    # Coca Cola: a director is a name whose next line is "Director".
    # Add that name. Manning is already captured from her /s/ line.

    # ALL CAPS names: keep the normally printed line underneath
    # (McCarthy, not Mccarthy. Chávez, not Chavez).

    # "Robert E. Moritz, Jr." is one person. Remove the comma before
    # Jr./Sr./II/III, then join the names with commas.

    signers = []
    for name in names:
        if name not in signers:
            signers.append(name)
    return ", ".join(signers)
```

On this version every `/s/` name came through. The LLP filter already kept out `/s/ Ernst & Young LLP` (Walmart, Coca-Cola) and `/s/ Deloitte & Touche LLP` (Procter & Gamble); those lines sit in the auditor's report, before `SIGNATURES`. Coca-Cola stopped at the four `/s/` names. Walmart's joined string still contained `Robert E. Moritz, Jr.`, and splitting that string on commas produced `Robert E. Moritz` and `Jr.`

#### 2. After Claude Code

Claude Code replaced the three comments. The Director loop it wrote is the one in the notebook: walk the lines, and when the next line starts with `Director`, take the last cell of the current line. It used a new `plain_letters` helper to drop duplicate names, and `str.title()` to fix the capitals. The two helpers came back like this:

```python
import unicodedata

def plain_letters(name):
    stripped = unicodedata.normalize("NFKD", name)
    stripped = "".join(ch for ch in stripped if not unicodedata.combining(ch))
    return re.sub(r"[^a-z]", "", stripped.lower())

def drop_comma_before_suffix(name):
    return re.sub(r",\s*(Jr\.|Sr\.|II|III)$", r" \1", name, flags=re.I)

# once the raw name was stripped:
name = drop_comma_before_suffix(name)
if name == name.upper():
    name = name.title()
```

The notebook then ran from top to bottom and wrote `legalproceedings.csv`. Apple's ten names matched the signature page, so the excerpt looked finished.

#### 3. After we read it

The Director loop is `split`, a slice, and `append`, so we kept it.

`unicodedata.normalize` and the `\1` backreference were not something we could trace the way we trace the `\u` table in section 2. We replaced both. Accents are folded with the same kind of replacement table as the HTML entities, and only so that two spellings of one person can be compared. The comma is removed by slicing. That is the helper in the notebook:

```python
def drop_comma_before_suffix(name):
    "'Moritz, Jr.' -> 'Moritz Jr.' without using a backreference."
    for suffix in ["Jr.", "Sr.", "II", "III"]:
        tail = ", " + suffix
        if len(name) >= len(tail) and name.lower()[-len(tail):] == tail.lower():
            original_suffix = name[-len(suffix):]
            return name[:-len(tail)] + " " + original_suffix
    return name
```

We left `name.title()` in place. It is a string method, and on Alphabet it already turned `ROGER W. FERGUSON JR.` into `Roger W. Ferguson Jr.`

#### 4. After we checked it against the filings

We copied the fiscal year end, the signature dates, and every signer from the ten filings into `manual`, then ran the check table.

Walmart now stored `Robert E. Moritz Jr.`, one name. Coca-Cola stored all 15. We re-opened both signature pages and counted. The comma rewrite had kept Ferguson and Moritz, and the Director rule had picked up Herb Allen through Amity Millhiser without also picking up `Attorney-in-fact` or the date lines.

Two names were still wrong, and a lower-case comparison was hiding them. `"CHRISTINE M. MCCARTHY".title()` is `Christine M. Mccarthy`. `"R. MARTIN CHAVEZ".title()` is `R. Martin Chavez`. The filing prints `Christine M. McCarthy` and `R. Martin Chávez` on the following line. We changed the all-caps branch to the version in the notebook: search the cells for that printed line, and call `.title()` only when no mixed-case cell exists. `plain_letters` is what lets `MCCARTHY` match `McCarthy`, and `CHAVEZ` match `Chávez`. The parentheses around P&G's printed name are stripped when the cells are built. The string we store is the printed one. `same_names` compares those strings as printed, so a wrong capital or a missing accent is a mismatch.

```python
if name == name.upper():
    printed = [c for c in cells if c != c.upper() and plain_letters(c) == plain_letters(name)]
    if printed:
        name = printed[0]
    else:
        name = name.title()
```

We restarted the kernel and ran every cell. `Counter(check_df["match"])` was 30 matches, and the mismatch cell printed nothing. That same run is the Item 3 check in section 5(a): each extract starts in the body of Item 3 and stops before the Item 4 heading.

We have read, run and tested all code in this notebook and can explain every step.